
# UKRI FoR Classifier — POC Batch Inference (Primary + Fallback)

This notebook implements the current POC deployment flow:

1. Read the input file from S3 using `boto3`.
2. Validate the required input fields.
3. Use `ApplicationTitle` + `ApplicationSummary` as the model text.
4. Reject only rows where **both** title and summary are null/blank.
5. Run the **primary FoR Group model**.
6. Identify rows where the primary model predicts no category above threshold.
7. Run the **fallback FoR Division model only on those unresolved rows**.
8. Convert predictions to long output format:
   - primary model → 4-digit FoR Group `category_id`
   - fallback model → 2-digit FoR Division `category_id`
9. Add stakeholder output fields.
10. Validate, save locally, and optionally upload to S3.

The two model artefacts are each expected to be a **single `.joblib` file** containing the fitted vectorizer/model components, thresholds and multilabel binarizer.


## 1. Imports and environment checks

In [ ]:

from pathlib import Path
from datetime import datetime
import json
import os
import re
import sys
import logging

import boto3
import joblib
import numpy as np
import pandas as pd

# Existing project preprocessing module.
# It should provide preprocess_text_fields(df, text_fields, new_field_name, n_jobs, batch_size).
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "models").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
REJECTED_DIR = DATA_DIR / "rejected"
UNRESOLVED_DIR = DATA_DIR / "unresolved"
LOG_DIR = PROJECT_ROOT / "logs"

for d in [INPUT_DIR, OUTPUT_DIR, REJECTED_DIR, UNRESOLVED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_preparation import preprocess_text_fields

print("PROJECT_ROOT:", PROJECT_ROOT)


## 2. Configuration — update only this cell when paths change

In [ ]:

# ---------- S3 ----------
S3_BUCKET = "ddg-poc-data.store.stg-ukri.cloud"
S3_INPUT_KEY = "deployment_test_data/202608191125_FoR_FoRClassification_1.1.parquet"
S3_INPUT_PREFIX = "deployment_test_data/"
S3_OUTPUT_PREFIX = "test_output/"

# If S3_INPUT_KEY is None/blank, the notebook will locate the newest csv/parquet under S3_INPUT_PREFIX.
INPUT_SUFFIXES = (".csv", ".parquet")

# ---------- Models ----------
MAIN_MODEL_PATH = PROJECT_ROOT / "models" / (
    "lemma_stop_neg_scale_10_mindf_5_maxdf_90_maxfold_5_min_pos_50_pos_prior_50_"
    "fields_of_research_negative_scaling_tfidf.joblib"
)

FALLBACK_MODEL_PATH = PROJECT_ROOT / "models" / (
    "lemma_stop_neg_scale_10_mindf_5_maxdf_90_maxfold_5_min_pos_50_pos_prior_50_"
    "fields_of_research_negative_scaling_tfidf_division.joblib"
)

# ---------- Input schema ----------
ID_FIELDS = ["ApplicationID", "ApplicationOriginSource"]
MODEL_TEXT_FIELDS = ["ApplicationTitle", "ApplicationSummary"]

# ---------- Output metadata ----------
TAXONOMY_FILE_TOKEN = "FoR"
TAXONOMY_VALUE = "FieldsOfResearch"
MODEL_NAME = "FoRClassification"
MODEL_VERSION = "1.1"
SCORE_TYPE = "uncalibrated"
OUTPUT_EXTENSION = ".csv"

# ---------- Processing ----------
N_JOBS = 4
PREPROCESS_BATCH_SIZE = 250

# Keep False during notebook testing if you only want to create the local output.
UPLOAD_OUTPUT_TO_S3 = False

print("MAIN_MODEL_PATH:", MAIN_MODEL_PATH)
print("FALLBACK_MODEL_PATH:", FALLBACK_MODEL_PATH)


## 3. Logging

In [ ]:

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"for_inference_{run_stamp}.log"

logger = logging.getLogger("for_inference")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

fh = logging.FileHandler(log_path)
fh.setFormatter(formatter)
logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setFormatter(formatter)
logger.addHandler(sh)

logger.info("Inference notebook initialised")
logger.info("Project root: %s", PROJECT_ROOT)


## 4. S3 helpers and locate input object

In [ ]:

s3 = boto3.client("s3")

def find_latest_s3_object(s3_client, bucket, prefix, allowed_suffixes=(".csv", ".parquet")):
    paginator = s3_client.get_paginator("list_objects_v2")
    candidates = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith(tuple(s.lower() for s in allowed_suffixes)):
                candidates.append(obj)

    if not candidates:
        raise FileNotFoundError(f"No matching files under s3://{bucket}/{prefix}")

    return max(candidates, key=lambda x: x["LastModified"])["Key"]


selected_input_key = (
    S3_INPUT_KEY
    if S3_INPUT_KEY
    else find_latest_s3_object(s3, S3_BUCKET, S3_INPUT_PREFIX, INPUT_SUFFIXES)
)

print(f"s3://{S3_BUCKET}/{selected_input_key}")
logger.info("Selected S3 input: s3://%s/%s", S3_BUCKET, selected_input_key)


## 5. Download input file from S3

In [ ]:

local_input_path = INPUT_DIR / Path(selected_input_key).name
s3.download_file(S3_BUCKET, selected_input_key, str(local_input_path))

print("Downloaded to:", local_input_path)
logger.info("Downloaded input to %s", local_input_path)


## 6. Read input

In [ ]:

def read_input(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".parquet":
        return pd.read_parquet(file_path)
    raise ValueError(f"Unsupported input format: {suffix}")


df_raw = read_input(local_input_path)
print("Input shape:", df_raw.shape)
display(df_raw.head())


## 7. Validate input schema and construct application key

In [ ]:

required_columns = ID_FIELDS + MODEL_TEXT_FIELDS
missing_columns = [c for c in required_columns if c not in df_raw.columns]

if missing_columns:
    raise ValueError(f"Missing required input columns: {missing_columns}")

df = df_raw.copy()

# Keep IDs as strings without changing their source values.
# ApplicationID alone is not assumed to be unique.
df["_application_key"] = (
    df["ApplicationID"].astype("string").fillna("")
    + "||"
    + df["ApplicationOriginSource"].astype("string").fillna("")
)

print("Rows:", len(df))
print("Distinct ApplicationID + ApplicationOriginSource keys:", df["_application_key"].nunique())


## 8. Reject rows where BOTH title and summary are empty

In [ ]:

def is_blank(series: pd.Series) -> pd.Series:
    return series.isna() | series.astype("string").fillna("").str.strip().eq("")

both_text_missing = is_blank(df["ApplicationTitle"]) & is_blank(df["ApplicationSummary"])

df_rejected = df.loc[both_text_missing].copy()
df_rejected["rejected_reason"] = "TITLE_AND_SUMMARY_EMPTY"

df_valid = df.loc[~both_text_missing].copy().reset_index(drop=True)

print("Total rows:", len(df))
print("Valid for inference:", len(df_valid))
print("Rejected:", len(df_rejected))

if len(df_rejected):
    rejected_path = REJECTED_DIR / f"rejected_{run_stamp}.csv"
    df_rejected.to_csv(rejected_path, index=False)
    print("Rejected rows saved to:", rejected_path)


## 9. Preprocess ApplicationTitle + ApplicationSummary

In [ ]:

df_cleaned = preprocess_text_fields(
    df=df_valid.copy(),
    text_fields=MODEL_TEXT_FIELDS,
    new_field_name="PROCESSED_TEXT",
    n_jobs=N_JOBS,
    batch_size=PREPROCESS_BATCH_SIZE,
)

if "PROCESSED_TEXT" not in df_cleaned.columns:
    raise KeyError("preprocess_text_fields did not create PROCESSED_TEXT")

print("Cleaned shape:", df_cleaned.shape)
display(df_cleaned[ID_FIELDS + MODEL_TEXT_FIELDS + ["PROCESSED_TEXT"]].head())


## 10. Load and validate the two joblib model bundles

In [ ]:

def load_model_bundle(model_path: Path) -> dict:
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    bundle = joblib.load(model_path)

    if not isinstance(bundle, dict):
        raise TypeError(
            f"Expected model artefact to be a dict, got {type(bundle).__name__}: {model_path}"
        )

    required = {"vectorizer", "models", "thresholds", "mlb"}
    missing = required - set(bundle.keys())
    if missing:
        raise KeyError(f"{model_path.name} is missing model-bundle keys: {sorted(missing)}")

    if len(bundle["models"]) != len(bundle["mlb"].classes_):
        raise ValueError(
            f"{model_path.name}: number of models ({len(bundle['models'])}) "
            f"does not match number of classes ({len(bundle['mlb'].classes_)})"
        )

    thresholds = np.asarray(bundle["thresholds"]).reshape(-1)
    if len(thresholds) != len(bundle["mlb"].classes_):
        raise ValueError(
            f"{model_path.name}: thresholds ({len(thresholds)}) "
            f"do not match classes ({len(bundle['mlb'].classes_)})"
        )

    return bundle


primary_model = load_model_bundle(MAIN_MODEL_PATH)
fallback_model = load_model_bundle(FALLBACK_MODEL_PATH)

print("Primary classes:", len(primary_model["mlb"].classes_))
print("Fallback classes:", len(fallback_model["mlb"].classes_))


## 11. Generic inference helper

In [ ]:

def _positive_probability(model, X):
    '''
    Return probability for the positive class for a binary classifier.

    The current joblib bundle stores one classifier per label.
    '''
    p = np.asarray(model.predict_proba(X))

    if p.ndim == 1:
        return p

    if p.shape[1] == 1:
        # Defensive handling for a degenerate binary estimator.
        return p[:, 0]

    return p[:, 1]


def perform_inference(model_bundle: dict, cleaned_df: pd.DataFrame):
    X = model_bundle["vectorizer"].transform(cleaned_df["PROCESSED_TEXT"])

    probabilities = np.column_stack([
        _positive_probability(model, X)
        for model in model_bundle["models"]
    ])

    thresholds = np.asarray(model_bundle["thresholds"]).reshape(1, -1)
    predictions = (probabilities >= thresholds).astype(np.int8)

    expected_shape = (len(cleaned_df), len(model_bundle["mlb"].classes_))
    if probabilities.shape != expected_shape:
        raise ValueError(
            f"Probability matrix shape {probabilities.shape}; expected {expected_shape}"
        )

    return probabilities, predictions


## 12. Run PRIMARY Group-level model

In [ ]:

primary_probabilities, primary_predictions = perform_inference(
    primary_model,
    df_cleaned
)

df_cleaned["_primary_prediction_count"] = primary_predictions.sum(axis=1)

print("Primary probability shape:", primary_probabilities.shape)
print("Applications with >=1 primary prediction:",
      int((df_cleaned["_primary_prediction_count"] > 0).sum()))
print("Applications with no primary prediction:",
      int((df_cleaned["_primary_prediction_count"] == 0).sum()))


## 13. Identify unresolved primary records

In [ ]:

resolved_mask = df_cleaned["_primary_prediction_count"] > 0

df_primary_resolved = df_cleaned.loc[resolved_mask].copy()
df_primary_unresolved = df_cleaned.loc[~resolved_mask].copy()
df_primary_unresolved["unresolved_reason"] = "NO_PRIMARY_CATEGORY_ABOVE_THRESHOLD"

print("Resolved:", len(df_primary_resolved))
print("Unresolved → fallback:", len(df_primary_unresolved))


## 14. Run FALLBACK Division-level model only on unresolved rows

In [ ]:

fallback_probabilities = np.empty((0, len(fallback_model["mlb"].classes_)))
fallback_predictions = np.empty((0, len(fallback_model["mlb"].classes_)), dtype=np.int8)

if len(df_primary_unresolved) > 0:
    # The fallback model is trained using the same preprocessing principle,
    # so the existing PROCESSED_TEXT is reused.
    fallback_probabilities, fallback_predictions = perform_inference(
        fallback_model,
        df_primary_unresolved.reset_index(drop=True)
    )

    df_primary_unresolved["_fallback_prediction_count"] = fallback_predictions.sum(axis=1).tolist()
else:
    df_primary_unresolved["_fallback_prediction_count"] = []

print("Fallback rows processed:", len(df_primary_unresolved))
print("Fallback rows resolved:",
      int((df_primary_unresolved["_fallback_prediction_count"] > 0).sum())
      if len(df_primary_unresolved) else 0)
print("Fallback rows still unresolved:",
      int((df_primary_unresolved["_fallback_prediction_count"] == 0).sum())
      if len(df_primary_unresolved) else 0)


## 15. Convert PRIMARY multi-label predictions to long format

In [ ]:

def extract_numeric_category_id(label, digits: int) -> int:
    '''
    Extract the leading numeric taxonomy code.

    Examples:
      "3007: Forestry sciences" -> 3007 when digits=4
      "42: Health sciences"     -> 42   when digits=2
    '''
    match = re.match(rf"^\s*(\d{{{digits}}})\b", str(label))
    if not match:
        raise ValueError(
            f"Could not extract a {digits}-digit category code from label: {label!r}"
        )
    return int(match.group(1))


primary_rows = []

for row_pos, (_, source_row) in enumerate(df_cleaned.iterrows()):
    selected = np.flatnonzero(primary_predictions[row_pos] == 1)

    for class_idx in selected:
        primary_rows.append({
            "ApplicationID": source_row["ApplicationID"],
            "ApplicationOriginSource": source_row["ApplicationOriginSource"],
            "category_id": extract_numeric_category_id(
                primary_model["mlb"].classes_[class_idx], digits=4
            ),
            "score": float(primary_probabilities[row_pos, class_idx]),
            "prediction_source": "PRIMARY",
        })

df_primary_predictions = pd.DataFrame(
    primary_rows,
    columns=[
        "ApplicationID",
        "ApplicationOriginSource",
        "category_id",
        "score",
        "prediction_source",
    ],
)

display(df_primary_predictions.head(10))
print("Primary prediction rows:", len(df_primary_predictions))


## 16. Convert FALLBACK Division predictions to long format

In [ ]:

fallback_rows = []

if len(df_primary_unresolved) > 0:
    fallback_input = df_primary_unresolved.reset_index(drop=True)

    for row_pos, (_, source_row) in enumerate(fallback_input.iterrows()):
        selected = np.flatnonzero(fallback_predictions[row_pos] == 1)

        for class_idx in selected:
            fallback_rows.append({
                "ApplicationID": source_row["ApplicationID"],
                "ApplicationOriginSource": source_row["ApplicationOriginSource"],
                "category_id": extract_numeric_category_id(
                    fallback_model["mlb"].classes_[class_idx], digits=2
                ),
                "score": float(fallback_probabilities[row_pos, class_idx]),
                "prediction_source": "FALLBACK_DIVISION",
            })

df_fallback_predictions = pd.DataFrame(
    fallback_rows,
    columns=[
        "ApplicationID",
        "ApplicationOriginSource",
        "category_id",
        "score",
        "prediction_source",
    ],
)

display(df_fallback_predictions.head(10))
print("Fallback prediction rows:", len(df_fallback_predictions))


## 17. Persist rows still unresolved after fallback

In [ ]:

if len(df_primary_unresolved) > 0:
    fallback_input = df_primary_unresolved.reset_index(drop=True)
    still_unresolved_mask = fallback_predictions.sum(axis=1) == 0
    df_still_unresolved = fallback_input.loc[still_unresolved_mask].copy()
    df_still_unresolved["unresolved_reason"] = "NO_FALLBACK_DIVISION_ABOVE_THRESHOLD"
else:
    df_still_unresolved = df_primary_unresolved.copy()

print("Still unresolved after fallback:", len(df_still_unresolved))

if len(df_still_unresolved):
    unresolved_path = UNRESOLVED_DIR / f"unresolved_{run_stamp}.csv"
    df_still_unresolved.to_csv(unresolved_path, index=False)
    print("Unresolved rows saved to:", unresolved_path)


## 18. Combine primary and fallback predictions

In [ ]:

df_all_predictions = pd.concat(
    [df_primary_predictions, df_fallback_predictions],
    ignore_index=True,
)

print("Combined prediction rows:", len(df_all_predictions))
print("Unique predicted applications:",
      df_all_predictions[["ApplicationID", "ApplicationOriginSource"]].drop_duplicates().shape[0]
      if len(df_all_predictions) else 0)

display(df_all_predictions.head(10))


## 19. Build stakeholder output schema

In [ ]:

model_run_date = datetime.now().date()

df_output = df_all_predictions.copy()

df_output["model_run_date"] = model_run_date
df_output["score_type"] = SCORE_TYPE

# Required downstream output is Decimal(5,2).
df_output["score"] = pd.to_numeric(df_output["score"], errors="raise").round(2)

df_output["Taxonomy"] = TAXONOMY_VALUE
df_output["model_name"] = MODEL_NAME
df_output["model_version"] = MODEL_VERSION

FINAL_COLUMNS = [
    "ApplicationID",
    "ApplicationOriginSource",
    "model_run_date",
    "category_id",
    "score_type",
    "score",
    "Taxonomy",
    "model_name",
    "model_version",
]

df_output = df_output[FINAL_COLUMNS]

# Explicit types where pandas can represent the stakeholder requirements.
df_output["ApplicationID"] = df_output["ApplicationID"].astype("string")
df_output["ApplicationOriginSource"] = df_output["ApplicationOriginSource"].astype("string")
df_output["category_id"] = pd.to_numeric(df_output["category_id"], errors="raise").astype("int64")
df_output["score_type"] = df_output["score_type"].astype("string")
df_output["Taxonomy"] = df_output["Taxonomy"].astype("string")
df_output["model_name"] = df_output["model_name"].astype("string")
df_output["model_version"] = df_output["model_version"].astype("string")

display(df_output.head(10))
print(df_output.dtypes)


## 20. Validate final output

In [ ]:

expected_columns = FINAL_COLUMNS

assert list(df_output.columns) == expected_columns
assert df_output["ApplicationID"].notna().all()
assert df_output["ApplicationOriginSource"].notna().all()
assert df_output["category_id"].notna().all()
assert df_output["score"].notna().all()
assert df_output["score"].between(0, 1).all()
assert (df_output["score_type"] == SCORE_TYPE).all()
assert (df_output["Taxonomy"] == TAXONOMY_VALUE).all()
assert (df_output["model_name"] == MODEL_NAME).all()
assert (df_output["model_version"] == MODEL_VERSION).all()

# The same application/category combination should not be duplicated.
duplicate_mask = df_output.duplicated(
    subset=["ApplicationID", "ApplicationOriginSource", "category_id"],
    keep=False,
)
if duplicate_mask.any():
    display(df_output.loc[duplicate_mask].sort_values(
        ["ApplicationID", "ApplicationOriginSource", "category_id"]
    ))
    raise ValueError("Duplicate application/category rows detected in final output")

print("Final output validation passed.")
print("Rows:", len(df_output))
print("Unique applications:",
      df_output[["ApplicationID", "ApplicationOriginSource"]].drop_duplicates().shape[0]
      if len(df_output) else 0)


## 21. Save final output locally

In [ ]:

output_timestamp = datetime.now().strftime("%Y%m%d%H%M")
output_filename = f"{output_timestamp}_{TAXONOMY_FILE_TOKEN}{OUTPUT_EXTENSION}"
local_output_path = OUTPUT_DIR / output_filename

df_output.to_csv(local_output_path, index=False)

print("Output saved:", local_output_path)
logger.info("Final output saved locally: %s", local_output_path)


## 22. Optional S3 upload

In [ ]:

output_s3_key = f"{S3_OUTPUT_PREFIX.rstrip('/')}/{output_filename}"

if UPLOAD_OUTPUT_TO_S3:
    s3.upload_file(str(local_output_path), S3_BUCKET, output_s3_key)
    print(f"Uploaded to s3://{S3_BUCKET}/{output_s3_key}")
    logger.info("Uploaded output to s3://%s/%s", S3_BUCKET, output_s3_key)
else:
    print("S3 upload disabled for POC testing.")
    print("Set UPLOAD_OUTPUT_TO_S3 = True in the configuration cell when ready.")


## 23. Run summary

In [ ]:

summary = {
    "input_s3_uri": f"s3://{S3_BUCKET}/{selected_input_key}",
    "input_rows": int(len(df_raw)),
    "valid_rows_for_inference": int(len(df_valid)),
    "rejected_empty_text_rows": int(len(df_rejected)),
    "primary_resolved_applications": int((df_cleaned["_primary_prediction_count"] > 0).sum()),
    "sent_to_fallback": int(len(df_primary_unresolved)),
    "fallback_resolved_applications": (
        int((fallback_predictions.sum(axis=1) > 0).sum())
        if len(df_primary_unresolved) else 0
    ),
    "still_unresolved_after_fallback": int(len(df_still_unresolved)),
    "final_prediction_rows": int(len(df_output)),
    "final_unique_applications": (
        int(df_output[["ApplicationID", "ApplicationOriginSource"]].drop_duplicates().shape[0])
        if len(df_output) else 0
    ),
    "local_output": str(local_output_path),
    "s3_output_uri": (
        f"s3://{S3_BUCKET}/{output_s3_key}" if UPLOAD_OUTPUT_TO_S3 else None
    ),
}

print(json.dumps(summary, indent=2, default=str))
logger.info("Run summary: %s", summary)



## Notes for the next productionisation step

The notebook deliberately keeps orchestration explicit for POC testing. Once the end-to-end run is confirmed in Ronin, the same functions can be moved into Python modules and called from a batch entry point/container.

Key behaviour already enforced here:

- `ApplicationID + ApplicationOriginSource` is treated as the application identity.
- Only `ApplicationTitle + ApplicationSummary` are used for prediction.
- Either title or summary may be null; a row is rejected only if both are empty.
- Primary predictions use the 4-digit Group code.
- The fallback model is invoked **only** when the primary model gives no prediction above threshold.
- Fallback predictions use the 2-digit Division code.
- One application can therefore produce multiple rows in the final output.
- The final output includes model run date, score type, taxonomy, model name and version.
